In [10]:
pip install xlrd

Note: you may need to restart the kernel to use updated packages.


In [16]:
import pandas as pd

# Check the problematic file - try different header rows
file = "Dataset/Rdir_2011_23_MADHYA_PRADESH.xls"

# Try reading with no header
df_no_header = pd.read_excel(file, header=None)
print("Shape with no header:", df_no_header.shape)
print("First few rows:")
print(df_no_header.head())


Shape with no header: (54, 6)
First few rows:
      0      1    2     3               4   5
0   NaN    NaN  NaN   NaN             NaN NaN
1   NaN    NaN  NaN   NaN             NaN NaN
2   NaN    NaN  NaN   NaN             NaN NaN
3   NaN    NaN  NaN   NaN  Madhya Pradesh NaN
4  23.0  230.0  1.0  10.0        Sheopur  NaN


In [18]:
import pandas as pd
import glob

# Get all Excel files in current folder
files = glob.glob("Dataset/*.xls")

df_list = []
skipped_files = []

for file in files:
    try:
        temp = pd.read_excel(file)
        
        # Only process files with 8 columns (standard format)
        if len(temp.columns) == 8:
            # Standardize column names
            temp.columns = [
                "state_code", "state_name",
                "district_code", "district_name",
                "subdistrict_code", "subdistrict_name",
                "village_code", "village_name"
            ]
            df_list.append(temp)
        else:
            skipped_files.append(f"{file} ({len(temp.columns)} columns)")
    except Exception as e:
        skipped_files.append(f"{file} (Error: {str(e)[:50]})")

df = pd.concat(df_list, ignore_index=True)

print(f"✓ Loaded files: {len(files) - len(skipped_files)}")
print(f"✗ Skipped files: {len(skipped_files)}")
if skipped_files:
    print(f"  Skipped: {skipped_files}")
print(f"\nTotal rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")


✓ Loaded files: 28
✗ Skipped files: 1
  Skipped: ['Dataset\\Rdir_2011_23_MADHYA_PRADESH.xls (6 columns)']

Total rows: 462893
Total columns: 8


In [19]:
# Remove fake/placeholder rows
df = df[df["village_name"] != df["state_name"]]
df = df[df["village_name"] != df["district_name"]]
df = df[df["village_name"] != df["subdistrict_name"]]

# Clean village names — remove (6), (11) type numbers
df["village_name"] = df["village_name"].str.replace(r"\(\d+\)", "", regex=True).str.strip()

# Standardize text
for col in ["state_name", "district_name", "subdistrict_name", "village_name"]:
    df[col] = df[col].str.strip().str.title()

# Remove duplicates and nulls
df = df.drop_duplicates()
df = df.dropna()

print("Rows after cleaning:", df.shape[0])
print("Null check:\n", df.isnull().sum())

Rows after cleaning: 455702
Null check:
 state_code          0
state_name          0
district_code       0
district_name       0
subdistrict_code    0
subdistrict_name    0
village_code        0
village_name        0
dtype: int64


In [20]:
# Split into relational tables
states = df[["state_code", "state_name"]].drop_duplicates().reset_index(drop=True)
districts = df[["district_code", "district_name", "state_code"]].drop_duplicates().reset_index(drop=True)
subdistricts = df[["subdistrict_code", "subdistrict_name", "district_code"]].drop_duplicates().reset_index(drop=True)
villages = df[["village_code", "village_name", "subdistrict_code"]].drop_duplicates().reset_index(drop=True)

# Export
states.to_csv("clean_states.csv", index=False)
districts.to_csv("clean_districts.csv", index=False)
subdistricts.to_csv("clean_subdistricts.csv", index=False)
villages.to_csv("clean_villages.csv", index=False)

print("✓ States:", len(states))
print("✓ Districts:", len(districts))
print("✓ Subdistricts:", len(subdistricts))
print("✓ Villages:", len(villages))

✓ States: 28
✓ Districts: 459
✓ Subdistricts: 5039
✓ Villages: 455702
